# 03 · Exploratory Data Analysis (EDA)

Before feature engineering and model training, we need to understand the data's characteristics,
distributions, and class-specific patterns.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter

df = pd.read_csv('../data/processed/dataset.csv')
print(f'Dataset: {len(df):,} messages  |  Ham: {(df.label=="ham").sum():,}  |  Spam: {(df.label=="spam").sum():,}')

## 1. Message Length Analysis

In [ ]:
df['char_len'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print('Character length by class:')
print(df.groupby('label')['char_len'].describe().round(1))
print(f'\nWord count by class:')
print(df.groupby('label')['word_count'].describe().round(1))

print(f'\nKey insight: Spam messages are on average {df[df.label=="spam"]["char_len"].mean() / df[df.label=="ham"]["char_len"].mean():.1f}x longer than ham messages')

## 2. URL and Link Analysis

In [ ]:
df['has_url'] = df['text'].str.contains(r'https?://|www\.', case=False, regex=True).astype(int)
df['url_count'] = df['text'].apply(lambda t: len(re.findall(r'https?://[^\s]+|www\.[^\s]+', t, re.I)))

url_by_class = df.groupby('label')['has_url'].mean() * 100
print('Percentage of messages containing URLs:')
print(f'  Ham:  {url_by_class.get("ham", 0):.1f}%')
print(f'  Spam: {url_by_class.get("spam", 0):.1f}%')

print(f'\nAvg URLs per message:')
print(df.groupby('label')['url_count'].mean().round(2))

## 3. Punctuation and Special Characters

In [ ]:
df['exclamation_count'] = df['text'].str.count('!')
df['question_count'] = df['text'].str.count(r'\?')
df['uppercase_ratio'] = df['text'].apply(lambda t: sum(c.isupper() for c in t) / max(len(t), 1))
df['special_char_count'] = df['text'].apply(lambda t: sum(not c.isalnum() and not c.isspace() for c in t))
df['has_currency'] = df['text'].str.contains(r'[$£€]|\b(?:usd|gbp|eur)\b', case=False, regex=True).astype(int)
df['has_phone'] = df['text'].str.contains(r'\+?\d[\d\s().-]{7,}\d', regex=True).astype(int)

features = ['exclamation_count', 'question_count', 'uppercase_ratio', 'special_char_count', 'has_currency', 'has_phone']
print('Feature means by class:')
print(df.groupby('label')[features].mean().round(3).T)

## 4. Most Common Words by Class

In [ ]:
def top_words(texts, n=20):
    words = []
    for t in texts:
        words.extend(re.findall(r'\b[a-z]{2,}\b', t.lower()))
    return Counter(words).most_common(n)

print('Top 20 words in HAM:')
for word, count in top_words(df[df.label == 'ham']['text']):
    print(f'  {word:15s} {count:5d}')

print(f'\nTop 20 words in SPAM:')
for word, count in top_words(df[df.label == 'spam']['text']):
    print(f'  {word:15s} {count:5d}')

## 5. Spam Indicator Terms

In [ ]:
spam_indicators = ['free', 'win', 'prize', 'claim', 'urgent', 'call', 'text', 'reply', 'stop', 'cash', 'offer', 'bonus', 'award', 'congratulations']

print('Spam indicator term frequency by class:')
for term in spam_indicators:
    ham_pct = df[df.label == 'ham']['text'].str.contains(term, case=False).mean() * 100
    spam_pct = df[df.label == 'spam']['text'].str.contains(term, case=False).mean() * 100
    ratio = spam_pct / max(ham_pct, 0.1)
    print(f'  {term:18s}  ham: {ham_pct:5.1f}%  spam: {spam_pct:5.1f}%  ratio: {ratio:5.1f}x')

## 6. Class Imbalance Summary

The dataset exhibits a natural class imbalance (~87% ham, ~13% spam). This is realistic for email environments but requires:
- **Stratified splitting** to preserve ratios across train/val/test
- **Class-weighted models** or oversampling during training
- **Evaluation beyond accuracy** (precision, recall, F1, ROC-AUC, PR-AUC)

In [ ]:
print('Class distribution:')
dist = df['label'].value_counts()
for label in ['ham', 'spam']:
    count = dist[label]
    pct = count / len(df) * 100
    print(f'  {label}: {count:,} ({pct:.1f}%)')

print(f'\nImbalance ratio: {dist["ham"] / dist["spam"]:.1f}:1')
print(f'\n✅ EDA complete. Key findings inform feature engineering in notebook 05.')